# HackODS UNAM 2026: Pipeline de Datos y Análisis Exploratorio (Equipo linuxitOS)

Este notebook contiene el pipeline ETL (Extract, Transform, Load) centralizado que alimenta el [Dashboard ODS 1 x ODS 6](https://github.com/linuxitOS-hackODS/linuxitOS-HackODS-UNAM/tree/main/dashboard).
    
El objetivo principal es demostrar la transparencia y reproducibilidad técnica de nuestro análisis de datos geoespaciales, sociodemográficos y financieros. Para facilitar la evaluación de nuestro repositorio sin perder la comprensión de nuestra arquitectura técnica, hemos consolidado la ejecución en un orquestador principal (`main.py`).

## Arquitectura del Pipeline ETL (7 Módulos)
Nuestro procesamiento de datos está dividido en 7 fases especializadas que se ejecutan de manera secuencial. A continuación se describe el propósito de cada script de la carpeta `scripts/`:

1. **`siods_extractor.py`**: Recolección de métricas de infraestructura municipal desde SIODS.
2. **`analyze_iter.py`**: Depuración demográfica y cálculo de ruralidad desde el ITER (INEGI).
3. **`clean_coneval.py`**: Normalización de Índices de Rezago Social y Pobreza (CONEVAL).
4. **`merge_inegi_coneval.py`**: Integración o *merge* de la base demográfica.
5. **`merge_shcp.py`**: Cruce financiero con el presupuesto federal (FAISMUN/FORTAMUN).
6. **`merge_conagua.py`**: Cruce hídrico de disponibilidad por regiones hidrológicas (CONAGUA).
7. **`simplify_geojson.py`**: Optimización algorítmica geoespacial (Douglas-Peucker) para reducir los polígonos de cuencas de 18MB a 1MB y garantizar alto rendimiento web.

## Ejecución del Orquestador
La celda de código a continuación corresponde a nuestro `main.py`, el cual invoca secuencialmente cada uno de estos módulos, construye el dataset analítico y lo exporta para consumo nativo en el framework Quarto.


In [ ]:
import subprocess
import sys
import os

def run_script(script_path):
    print(f"\n{'-'*50}")
    print(f"[INFO] Ejecutando módulo: {script_path}")
    print(f"{'-'*50}")
    try:
        # Usar subprocess para llamar al script de python
        result = subprocess.run([sys.executable, script_path], check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] Falla en la ejecución de {script_path}. Abortando pipeline.")
        sys.exit(1)
    except Exception as e:
        print(f"\n[FATAL] Error inesperado en el módulo {script_path}: {e}")
        sys.exit(1)

def main():
    print("Iniciando Pipeline Automatizado de Datos (ETL): HackODS UNAM 2026 - linuxitOS\n")
    
    # Ajuste dinámico del directorio de trabajo (soporte para Terminal vs Jupyter Notebooks)
    # Si Jupyter lanza la libreta desde la carpeta 'notebooks/', bajamos un nivel para poder ver 'scripts/'.
    if not os.path.exists('scripts') and os.path.exists(os.path.join('..', 'scripts')):
        os.chdir('..')
        print("[INFO] Entorno Jupyter Notebook detectado. El directorio de trabajo se ajustó a la raíz del proyecto.\n")
    elif not os.path.exists('scripts'):
         print("[ERROR] El entorno no pudo localizar la carpeta 'scripts/'.")
         print("        Asegúrate de ejecutar desde la raíz del repositorio o dentro de la carpeta 'notebooks/'.")
         sys.exit(1)

    # Orden secuencial de ejecución para la integración del dataset maestro
    pipeline = [
        "scripts/siods_extractor.py",
        "scripts/analyze_iter.py",
        "scripts/clean_coneval.py",
        "scripts/merge_inegi_coneval.py",
        "scripts/merge_shcp.py",
        "scripts/merge_conagua.py",
        "scripts/simplify_geojson.py"
    ]

    for script in pipeline:
        run_script(script)

    print("\n[ÉXITO] Pipeline ETL concluido satisfactoriamente.")
    print("[INFO] El dataset analítico 'final_merged_data.csv' ha sido generado exitosamente.")
    print("[INFO] Para compilar el dashboard Quarto, ejecute el siguiente comando:")
    print("       quarto render dashboard/index.qmd")

if __name__ == "__main__":
    main()

# Para ejecutar el pipeline completo desde esta libreta, 
# descomente la siguiente línea si el entorno es adecuado:
# main()


## Conclusión del Pipeline

Al ejecutar el orquestador `main.py`, se generan los datasets limpios y el archivo geoespacial optimizado en la carpeta `datos/`. Estos archivos son la columna vertebral y fuente de la verdad para nuestro tablero de Quarto.

El archivo analítico final consolidado es: `datos/final_merged_data.csv`.
